In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import math
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# ==========================================================
# USER PARAMETERS
# ==========================================================

N_QUBITS = 10
MARKED_STATE = "1011010101"        # Must have length = N_QUBITS
SHOTS = 32768

USE_SIMULATOR = False  # Set to False to run on a real device

IBM_API_KEY = os.getenv("IBM_API_KEY")

# ==========================================================
# VALIDATION
# ==========================================================

if len(MARKED_STATE) != N_QUBITS:
    raise ValueError("Length of MARKED_STATE must equal N_QUBITS.")

# Reverse because Qiskit uses little-endian ordering
TARGET = MARKED_STATE[::-1]

# ==========================================================
# ORACLE
# ==========================================================

def oracle(qc, target):

    n = len(target)

    # Convert target state into |111...1>
    for i, bit in enumerate(target):
        if bit == "0":
            qc.x(i)

    # Phase flip
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    # Restore qubits
    for i, bit in enumerate(target):
        if bit == "0":
            qc.x(i)


# ==========================================================
# DIFFUSER
# ==========================================================

def diffuser(qc, n):

    qc.h(range(n))
    qc.x(range(n))

    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    qc.x(range(n))
    qc.h(range(n))


# ==========================================================
# BUILD GROVER CIRCUIT
# ==========================================================

qc = QuantumCircuit(N_QUBITS)

# Uniform superposition
qc.h(range(N_QUBITS))

# Optimal iterations
iterations = max(
    1,
    int((math.pi / 4) * math.sqrt(2 ** N_QUBITS))
)

print(f"Using {iterations} Grover iterations")

for _ in range(iterations):

    oracle(qc, TARGET)

    diffuser(qc, N_QUBITS)

qc.measure_all()

# ==========================================================
# CONNECT TO IBM QUANTUM
# ==========================================================

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=IBM_API_KEY
)

if USE_SIMULATOR:

    backend = service.least_busy(
        simulator=True
    )

else:

    backend = service.least_busy(
        operational=True,
        simulator=False
    )

print("Backend:", backend.name)

# ==========================================================
# TRANSPILATION
# ==========================================================

transpiled = transpile(
    qc,
    backend=backend,
    optimization_level=3
)

# ==========================================================
# EXECUTION
# ==========================================================

sampler = Sampler(mode=backend)

job = sampler.run(
    [transpiled],
    shots=SHOTS
)

print("Job ID:", job.job_id())

result = job.result()

counts = result[0].data.meas.get_counts()

# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\nMeasurement Counts\n")

sorted_counts = sorted(
    counts.items(),
    key=lambda x: x[1],
    reverse=True
)

for state, count in sorted_counts:

    marker = ""

    if state == MARKED_STATE:
        marker = " <--- TARGET"

    print(f"{state} : {count}{marker}")

qiskit_runtime_service._discover_account:WARNING:2026-08-03 01:56:50,509: Loading account with the given token. A saved account will not be used.


Using 25 Grover iterations


qiskit_runtime_service.__init__:WARNING:2026-08-03 01:56:55,012: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-08-03 01:56:55,670: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-08-03 01:57:00,100: Using instance: open-instance, plan: open


Backend: ibm_marrakesh
Job ID: d9nqg8gqs0bc73e3oelg

Measurement Counts

0000101101 : 64
0010000110 : 64
0000000110 : 64
0010000111 : 62
0000001110 : 61
1000000110 : 60
0000010111 : 59
0000100010 : 58
0000111101 : 56
0100101110 : 55
0000001100 : 55
0000100100 : 54
0010100110 : 54
0000010000 : 54
0000100001 : 54
0110000010 : 53
0010011110 : 53
0010001110 : 53
0000010100 : 53
0000100110 : 52
0000000000 : 52
0100000111 : 52
0100001111 : 52
1000000100 : 52
0100100110 : 52
0010000011 : 51
0010110100 : 51
0000000100 : 51
0100101111 : 51
0010011100 : 50
1000111111 : 50
0000001011 : 50
0100100000 : 49
0000000011 : 49
0000101011 : 49
0000000111 : 49
1000111110 : 49
0010010000 : 48
0000100000 : 48
0001000101 : 48
1100001100 : 48
0000011110 : 48
1100101110 : 48
0000011111 : 48
0110111011 : 48
0000000001 : 47
0000110000 : 47
0100010111 : 47
0010010100 : 47
1000000101 : 47
1100000110 : 47
1000001110 : 47
0100101100 : 47
0000110110 : 47
1110101111 : 47
0100100100 : 47
0110010111 : 47
0010001011 : 47